In [3]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# import torch_xla.core.xla_model as xm #when using tpu

#hyperparameters
batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layers = 4
dropout = 0.0

torch.manual_seed(1337);

In [8]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [54]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [55]:
x,y = get_batch('train')
x.shape, y.shape

(torch.Size([16, 32]), torch.Size([16, 32]))

In [56]:
decode(x[0].tolist()), decode(y[0].tolist())

("The one part suffer'd, the other", "he one part suffer'd, the other ")

In [5]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [13]:
class Head(nn.Module):
    """ one head of the self attention """
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B,T,C = x.shape

        k = self.key(x) # (B,T,C)
        q = self.query(x) #(B,T,C)
        #affinities
        wei = q @ k.transpose(-2,-1) * C ** -0.5   # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))         # (B, T, T)
        wei = F.softmax(wei,dim=-1) # (B, T, T)
        wei = self.dropout(wei)

        v = self.value(x) # (B,T,C)
        out = wei @ v
        return out
    
class MultiHeadAttention(nn.Module):
        """ multiple heads of self-attention in parallel"""
        def __init__(self, num_heads, head_size):
            super().__init__()
            self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
            self.proj = nn.Linear(n_embd, n_embd)
            self.dropout = nn.Dropout(dropout)
        
        def forward(self, x):
            out = torch.cat([h(x) for h in self.heads], dim=-1)
            out = self.dropout(self.proj(out))
            return out 
    
class FeedForward(nn.Module):
        """linear layer followed by a non linearity """
        def __init__(self, n_embd):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_embd, 4 * n_embd),
                nn.ReLU(), 
                nn.Linear(4*n_embd, n_embd),
                nn.Dropout(dropout)
            )
        
        def forward(self, x):
            return self.net(x)
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    
    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


In [16]:
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters()))

211841


In [17]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [18]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.3040, val loss 4.3084
step 100: train loss 2.6699, val loss 2.6748
step 200: train loss 2.5066, val loss 2.5105
step 300: train loss 2.4056, val loss 2.4083
step 400: train loss 2.3406, val loss 2.3390
step 500: train loss 2.2999, val loss 2.3104
step 600: train loss 2.2386, val loss 2.2470
step 700: train loss 2.2106, val loss 2.2144
step 800: train loss 2.1726, val loss 2.1914
step 900: train loss 2.1318, val loss 2.1586
step 1000: train loss 2.1099, val loss 2.1444
step 1100: train loss 2.0689, val loss 2.1021
step 1200: train loss 2.0435, val loss 2.0835
step 1300: train loss 2.0172, val loss 2.0733
step 1400: train loss 1.9928, val loss 2.0536
step 1500: train loss 1.9780, val loss 2.0382
step 1600: train loss 1.9530, val loss 2.0216
step 1700: train loss 1.9427, val loss 2.0045
step 1800: train loss 1.9069, val loss 1.9990
step 1900: train loss 1.9055, val loss 1.9823
step 2000: train loss 1.8829, val loss 1.9758
step 2100: train loss 1.8642, val loss 1.9684


In [19]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))



Clown:
Stay cown, which yet that sen beaut we and Our my callea
My Lordt he us he heat? we that a enarched you,
Do acrizong he owns, tof it he covericeding,
We mire, send, will is that Got Say.

WARGILG:
Andward!

AUFIDIUS:
The huse on hir spect; and thee yean's and nor
To tell I with
this demptinglect in on her eights eye more rive
And him he poor of his but that none firs so;
Anging me mone youled at to fear out, Some,
He but hasting is a adave the Edward huth cour?

ANGELO:
God more my care,
One to Pome, good of 'twer me?
 so, I wear to poor between threand thou suciles thy welcond,
Thou confessy that that lord
Is as detter work.

Luet Peeber:
To to dedish commans couner?
Youie so upon sir! 'e it as ny, come,
Against is housed, I cons, out as and back with
were but we me. I with self monder so lack.

Prived:
Fissicious; let it his divine noth, though is that is is
onter no come to me lip.

GLOUCEMBYCENCE:
Do must telll welk thou so hate apoin.

BENBIUS:
Had, there for Gly, in give